In [1]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
sys.path.append(str(project_root))

In [ ]:
import pandas as pd
from src.features import compute_rsi, compute_moving_average, compute_volatility
from src.model import time_series_split, train_logistic_regression, train_random_forest, evaluate_model




df = pd.read_csv("../data/raw/AAPL.csv", index_col=0)

# FIX: ensure Close is numeric
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

df["rsi_14"] = compute_rsi(df["Close"], 14)
df["ma_10"] = compute_moving_average(df["Close"], 10)
df["ma_20"] = compute_moving_average(df["Close"], 20)
df["volatility_10"] = compute_volatility(df["Close"], 10)

df.head(20)


XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/ankandatta/Documents/Projects/Development/stock_price_trend_classifier/env/lib/python3.9/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <89AD948E-E564-3266-867D-7AF89D6488F0> /Users/ankandatta/Documents/Projects/Development/stock_price_trend_classifier/env/lib/python3.9/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp

In [ ]:
df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

df[["Close", "target"]].head(10)

,Close,target
Price,,
Ticker,NaN,0
Date,NaN,0
2024-01-02,183.903244,0
2024-01-03,182.526230,0
2024-01-04,180.208115,0
2024-01-05,179.484940,1
2024-01-08,183.823975,0
2024-01-09,183.407913,1
2024-01-10,184.448074,0


In [ ]:
df_final = df.dropna().copy()


feature_cols = ["rsi_14", "ma_10", "ma_20", "volatility_10"]
X = df_final[feature_cols]
y = df_final["target"]

In [ ]:
X = df_final[feature_cols]
y = df_final["target"]


In [ ]:
X.head()

,rsi_14,ma_10,ma_20,volatility_10
Price,,,,
2024-01-30,55.832606,189.490460,186.132671,0.014680
2024-01-31,46.739534,189.660852,186.071249,0.016116
2024-02-01,52.166515,189.485507,186.200529,0.012728
2024-02-02,49.883310,188.919852,186.395687,0.011381
2024-02-05,56.857444,188.304659,186.717647,0.011058


In [ ]:
y.head()

Price
2024-01-30    0
2024-01-31    1
2024-02-01    0
2024-02-02    1
2024-02-05    1
Name: target, dtype: int64

In [ ]:
y.value_counts(normalize=True)


target
1    0.546748
0    0.453252
Name: proportion, dtype: float64

In [ ]:
X_train, X_test, y_train, y_test = time_series_split(X, y)

model, scaler = train_logistic_regression(X_train, y_train)

accuracy, report = evaluate_model(
    model,
    scaler,
    X_test,
    y_test
)

print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(report)


Accuracy: 0.5454545454545454

Classification Report:

              precision    recall  f1-score   support

           0       0.51      0.39      0.44        46
           1       0.56      0.68      0.62        53

    accuracy                           0.55        99
   macro avg       0.54      0.54      0.53        99
weighted avg       0.54      0.55      0.54        99



In [ ]:
X_train.shape, X_test.shape


((393, 4), (99, 4))

In [ ]:
y_train.shape, y_test.shape


((393,), (99,))

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.5858585858585859

Classification Report:

              precision    recall  f1-score   support

           0       0.56      0.54      0.55        46
           1       0.61      0.62      0.62        53

    accuracy                           0.59        99
   macro avg       0.58      0.58      0.58        99
weighted avg       0.59      0.59      0.59        99



In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model_scaled = LogisticRegression(max_iter=1000)
model_scaled.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred_scaled = model_scaled.predict(X_test_scaled)

from sklearn.metrics import accuracy_score, classification_report

print("Scaled Accuracy:", accuracy_score(y_test, y_pred_scaled))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_scaled))


Scaled Accuracy: 0.5454545454545454

Classification Report:

              precision    recall  f1-score   support

           0       0.51      0.39      0.44        46
           1       0.56      0.68      0.62        53

    accuracy                           0.55        99
   macro avg       0.54      0.54      0.53        99
weighted avg       0.54      0.55      0.54        99



In [ ]:
# Time-based split (same split for fair comparison)
X_train, X_test, y_train, y_test = time_series_split(X, y)

# Train RandomForest
rf_model = train_random_forest(X_train, y_train)

# Evaluate RandomForest
from sklearn.metrics import accuracy_score, classification_report

y_pred_rf = rf_model.predict(X_test)

print("RandomForest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nRandomForest Classification Report:\n")
print(classification_report(y_test, y_pred_rf))


NameError: name 'time_series_split' is not defined